<!-- GENERATED by scripts/build_notebooks.py from _variables.yml. Do not edit this cell. -->
# 3 · Model-free prediction and control

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/practical-rl/blob/main/notebooks/03-model-free.ipynb)

**Practical Reinforcement Learning** · Day 1 · Module 3 · briefing 14:45, lab 15:25 · core 50 min · designed for Google Colab, CPU runtime

Learn values and policies from experience with Monte Carlo, TD, SARSA and Q-learning.

**By the end of this notebook you can:**

- Measure the RMS error, squared bias and variance of MC and TD(0) estimates over 10 seeds against the exact V.
- Implement SARSA and Q-learning and explain the cliff contrast under a constant epsilon.
- Diagnose a terminal-bootstrap bug from the Q-values and a value curve, and say why the return curve does not show it.
- Compute n-step TD targets and place them between MC and TD.

**How the lab works.** Each exercise: *Predict* (write your guess first) → a `# TODO` cell you complete → *Run* → *Explain* → a checkpoint that passes or fails. A folded Hint and Solution sit under each TODO. The solution never replaces your code; if you are stuck, run `lab.use_reference(N)` to continue with it.

Briefing: [Module 3 · In the lab](https://project-delphi.github.io/practical-rl/modules/03-model-free.html#in-the-lab)

**Next:** [Module 4 · Bandits from first principles](https://project-delphi.github.io/practical-rl/modules/04-bandits.html)

In [ ]:
#@title Setup: run this cell first { display-mode: "form" }
# GENERATED by scripts/build_notebooks.py. On Colab it installs exactly what this lab adds
# (keeping every preinstalled package); elsewhere it checks versions and warns.
import importlib.metadata as _md
import os as _os
import subprocess as _sp
import sys as _sys

PRL_REF = "main"  #@param {type:"string"}
WORKED = False  #@param {type:"boolean"}
SEED = 0  #@param {type:"integer"}

_PINS = {"gymnasium": "1.4.0"}
_ALLOW = ["gymnasium"]
_CORE = ["torch", "numpy", "scipy", "pandas", "matplotlib", "transformers", "accelerate", "datasets"]
_PRL_SPEC = _os.environ.get("PRL_SPEC") or "git+https://github.com/project-delphi/practical-rl@" + PRL_REF + "#subdirectory=prl"
_on_colab = ("google.colab" in _sys.modules or bool(_os.environ.get("COLAB_RELEASE_TAG"))
             or _os.environ.get("PRL_PLATFORM") == "colab-sim")


def _norm(name):
    return name.lower().replace("_", "-").replace(".", "-")


def _installed_pins(names=None):
    """One name==version per package: the copy Python imports (first on sys.path).

    Colab's sys.path also holds older system copies of some packages; pinning
    every copy would ask pip for two versions of the same package.
    """
    pins = {}
    for d in _md.distributions():
        name = d.metadata["Name"]
        key = _norm(name) if name else None
        if not key or key in pins or key in _keep or (names is not None and key not in names):
            continue
        try:
            pins[key] = f"{name}=={_md.version(name)}"
        except _md.PackageNotFoundError:
            continue
    return sorted(pins.values())


def _pip_install(constraints):
    with open("/tmp/prl-constraints.txt", "w") as fh:
        fh.write("\n".join(constraints) + "\n")
    cmd = [_sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/prl-constraints.txt",
           *[f"{k}=={val}" for k, val in _PINS.items()], _PRL_SPEC]
    return _sp.run(cmd, capture_output=True, text=True)


if _on_colab:
    _keep = set(_ALLOW) | {"prl"}
    _before = {k: _md.version(k) for k in ("numpy", "torch", "scipy") if k in _sys.modules}
    _r = _pip_install(_installed_pins())
    if _r.returncode != 0:
        # Fall back to protecting only the heavy preinstalled packages.
        print("Keeping every preinstalled package was not possible; keeping the core ones "
              f"({', '.join(_CORE)}) and retrying.")
        print(_r.stderr.strip().splitlines()[-1] if _r.stderr.strip() else "")
        _r = _pip_install(_installed_pins(set(_CORE)))
    if _r.returncode != 0:
        print(_r.stdout[-3000:], _r.stderr[-3000:])
        raise SystemExit("Setup failed: the runtime's packages changed. Tell the instructor.")
    if any(_md.version(k) != val for k, val in _before.items()):
        print("A package that was already imported changed version: Runtime → Restart session, "
              "then run this cell again.")
else:
    for _name, _want in _PINS.items():
        try:
            _have = _md.version(_name)
        except _md.PackageNotFoundError:
            _have = None
        if _have != _want:
            print(f"Warning: this lab expects {_name} {_want}, found {_have}. "
                  "Run `uv sync --group notebooks` in the repository.")

import prl
import prl.plot
import prl.runtime

prl.runtime.print_versions(extra=list(_PINS))
prl.plot.style()
print(f"SEED = {SEED}")

In [ ]:
# GENERATED by scripts/build_notebooks.py. Starts the lab harness; rerunning it is safe.
import prl.lab

lab = prl.lab.init("03-model-free", "14abfe2a9f869c59", designed="colab-cpu", api=1, ns=globals(),
                   checkpoints=["1.1", "1.2", "2a.1", "2a.2", "2b", "3", "4", "5.1", "5.2", "6"], packages=["gymnasium"], seat=SEED, badge_ref=PRL_REF,
                   worked=WORKED or None)

In this lab the model is gone. You estimate the value of a fixed policy from sampled
episodes twice, by Monte Carlo and by TD(0), and measure the bias and variance of each
against the exact answer from Lab 1. Then you learn to act: SARSA and Q-learning on the
4 × 12 cliff, where the two methods walk different paths. You finish by finding a bug in
someone else's Q-learning that only an optimistic start can reveal.

The briefing's equations are named in brackets, for example (MC) or (TD(0)). Its table
"Equations and where you implement them" gives each name's equation number.

In [ ]:
from typing import NamedTuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from prl import envs
from prl.checks import m03

ROWS, COLS, GAMMA = 4, 12, 0.95  # the cliff of Module 1's figure, discounted
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3

envs.register()
env = gym.make("prl/CliffGridworld-v0", rows=ROWS, cols=COLS, gamma=GAMMA)
mdp = env.unwrapped.mdp  # the same P, R and gamma you wrote as arrays in Lab 1
S, A = mdp.n_states, mdp.n_actions
START, GOAL = mdp.start, mdp.goal
ABOVE_GOAL = GOAL - COLS  # the cell whose "down" is the final move of every route

# Budgets. Lab code reads them from here and never branches on QUICK. The control budget is
# the same in both: the design scan found the learners' greedy paths unreliable below about
# 1000 episodes at alpha 0.1, and 1000 episodes cost well under a second of compute.
budget = lab.use_settings(
    prl.runtime.settings(
        dict(
            episodes=1000, alpha=0.05, control_episodes=1000, control_alpha=0.1, window=500
        ),  # live
        dict(
            episodes=200, alpha=0.05, control_episodes=1000, control_alpha=0.1, window=500
        ),  # QUICK
        designed="colab-cpu",
    )
)
EPSILON = 0.1  # the exploration rate, constant throughout (Day 2 is about doing better)
print(dict(budget))

In [ ]:
# Reference code included from labs/shared/evaluate_exact.py.
# origin: labs/src/01-mdps-bellman.py
# The reference version of your Lab 1 code (Exercise 4): exact policy evaluation.
def evaluate_exact(P, R, gamma, pi):
    """V^pi from (I - gamma P_pi) V = r_pi, where pi is an (S, A) matrix of probabilities."""
    P_pi = np.einsum("sa,sat->st", pi, P)
    r_pi = (pi * R).sum(axis=1)
    return np.linalg.solve(np.eye(len(r_pi)) - gamma * P_pi, r_pi)

### The policy, the runner and the ground truth

Prediction needs a fixed policy whose episodes end. The **ε-soft safe route** takes the
route action (up the left column, right along the top row, down the right column) with
probability 1 − ε, and a uniformly random action otherwise. Its exact value comes from
Lab 1's `evaluate_exact`, included above as the reference version of your code.

The runner uses the Gymnasium environment, so every transition carries `terminated`
and `truncated` separately. Errors are measured on the 17 states of the route, which
the policy visits on nearly every episode.

In [ ]:
def safe_route_policy(epsilon):
    """(S, A) action probabilities of the epsilon-soft safe route."""
    pi = np.full((S, A), epsilon / A)
    for s in range(S):
        r, c = divmod(s, COLS)
        route_action = DOWN if c == COLS - 1 else (RIGHT if r == 0 else UP)
        pi[s, route_action] += 1.0 - epsilon
    return pi


def route_states(pi):
    """The states the route visits when no random move happens, start first, goal excluded."""
    s, route = START, [START]
    while s != GOAL:
        s = int(np.argmax(mdp.P[s, int(np.argmax(pi[s]))]))
        route.append(s)
    return route[:-1]


class Episode(NamedTuple):
    states: list  # s_0, ..., s_T
    actions: list  # a_0, ..., a_{T-1}
    rewards: list  # r_0, ..., r_{T-1}
    terminated: bool  # s_T is a terminal state
    truncated: bool  # the episode was cut by the step limit; s_T is not terminal


def run_episode(env, pi, rng):
    """One episode under the policy matrix pi, with explicit terminated / truncated flags."""
    s, _ = env.reset()
    states, actions, rewards = [int(s)], [], []
    while True:
        a = int(rng.choice(A, p=pi[s]))
        s, r, terminated, truncated, _ = env.step(a)
        actions.append(a)
        rewards.append(float(r))
        states.append(int(s))
        if terminated or truncated:
            return Episode(states, actions, rewards, bool(terminated), bool(truncated))


def collect(env, pi, n, seed):
    """n episodes under pi; all randomness comes from seed."""
    env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    return [run_episode(env, pi, rng) for _ in range(n)]


def rms_on_route(V):
    """Root-mean-square error of V against the exact values, over the route states."""
    return float(np.sqrt(np.mean((V[ROUTE] - V_EXACT[ROUTE]) ** 2)))


PI = safe_route_policy(EPSILON)
ROUTE = route_states(PI)
V_EXACT = evaluate_exact(mdp.P, mdp.R, GAMMA, PI)
P_pi = np.einsum("sa,sat->st", PI, mdp.P)
alive = [s for s in range(S) if s != GOAL]  # P(episode longer than n steps), exactly from P_pi
tail = np.zeros(len(alive))
tail[alive.index(START)] = 1.0
tail = tail @ np.linalg.matrix_power(P_pi[np.ix_(alive, alive)], 50)
print(f"Route: {len(ROUTE)} states; exact V(start) = {V_EXACT[START]:.3f}")
print(
    f"Return of a clean episode on the safe route (17 moves): {-(1 - GAMMA**17) / (1 - GAMMA):.3f}"
)
print(f"P(episode longer than 50 steps) = {tail.sum():.1e}; the step limit is 500, so no")
print("episode is truncated in practice, and the returns are complete (the briefing says why).")

## Exercise 1 · First-visit Monte Carlo prediction (9 min)

**First-visit Monte Carlo.** The return G_t after a visit to s is an unbiased sample of
V^π(s). First-visit MC averages, for each state, the return after its first visit in each
episode (MC). Work backward through each episode with G = r_t + γ G, then average.

**Predict.** Every episode starts at the start cell, and a fall sends the agent back there. After N
episodes, how many returns does first-visit MC average for the start: N, more than N, or
fewer? Will your first episode's return be a clean episode's −11.64 (printed above), or
lower?

In [ ]:
def mc_prediction(episodes, gamma, n_states):
    """First-visit Monte Carlo: V[s] = mean return after the first visit to s in each episode.

    Each episode has states s_0..s_T, rewards r_0..r_{T-1}; s_T is never updated.
    """
    V, N = np.zeros(n_states), np.zeros(n_states)
    # TODO 1: for each episode, compute G_t for every t from the end backward, then for each
    # state's FIRST visit add its return to the running mean (keep the count in N).
    raise NotImplementedError(
        "TODO 1: write mc_prediction, or run lab.use_reference(1) to continue with the solution"
    )

<details><summary>Hint 1</summary>

Two passes per episode. Backward: `G = r + gamma * G` fills a list of returns, one per
time step. Forward: a `set` of states seen so far in this episode tells you which visit is
the first. The running mean is `V[s] += (G_t - V[s]) / N[s]` after `N[s] += 1`.

</details>

In [ ]:
#@title Solution 1 · mc_prediction (open after trying) { display-mode: "form" }
@lab.solution(1)
def mc_prediction(episodes, gamma, n_states):
    """First-visit Monte Carlo: V[s] = mean return after the first visit to s in each episode."""
    V, N = np.zeros(n_states), np.zeros(n_states)
    for ep in episodes:
        G, returns = 0.0, [0.0] * len(ep.rewards)
        for t in reversed(range(len(ep.rewards))):
            G = ep.rewards[t] + gamma * G
            returns[t] = G
        seen = set()
        for t, s in enumerate(ep.states[:-1]):
            if s in seen:
                continue
            seen.add(s)
            N[s] += 1
            V[s] += (returns[t] - V[s]) / N[s]
    return V

In [ ]:
episodes = collect(env, PI, budget.episodes, seed=SEED)
lengths = [len(ep.rewards) for ep in episodes]
print(
    f"{len(episodes)} episodes; mean length {np.mean(lengths):.1f}, longest {max(lengths)}, "
    f"truncated: {sum(ep.truncated for ep in episodes)}"
)
first = episodes[0]
print(f"Return of your first episode: {mc_prediction([first], GAMMA, S)[START]:.3f}")
n_start = sum(START in ep.states[:-1] for ep in episodes)
print(f"First-visit returns averaged for the start: {n_start} (episodes: {len(episodes)})")
V_mc = mc_prediction(episodes, GAMMA, S)
mc_rms = rms_on_route(V_mc)
lab.metric("mc_rms", round(mc_rms, 4))
print(
    f"MC: V(start) = {V_mc[START]:.3f} (exact {V_EXACT[START]:.3f}); RMS on the route {mc_rms:.3f}"
)
fig, ax = plt.subplots()
ax.plot(V_EXACT[ROUTE], marker="o", label="exact (Lab 1)")
ax.plot(V_mc[ROUTE], marker="s", label=f"first-visit MC, {budget.episodes} episodes")
ax.set(xlabel="position along the route (0 = start)", ylabel="V", title="Monte Carlo on the route")
ax.legend()
plt.show()

**Explain.** Your first episode's return is one sample of V(start); the estimate is the average of all of
them (MC). The exact V(start), about −15.4, is below a clean episode's −11.64, the single most
likely return: which rare episodes pull the expectation down? And as the number of episodes
grows, what shrinks: the bias, the variance, or both?

In [ ]:
lab.check(1, m03.check_mc_prediction, mc_prediction, label="1.1")

In [ ]:
lab.check(1, m03.check_mc_rms, mc_prediction, label="1.2")

## Exercise 2a · TD(0) (6 min)

**TD(0).** Replace the tail of the return by the current estimate: the target is
r_t + γ V(s_{t+1}), and V(s_t) moves a fraction α of the way toward it (TD(0)). If
s_{t+1} is terminal its value is 0, so the target is r_t alone; if the episode was merely
truncated, keep the bootstrap (terminal mask). Process transitions in time order.

**Predict.** TD(0) with α = 0.05 starts from V = 0 and processes your first episode in time order.
Afterward, what is V at the cell directly above the goal, and what is V(start)? Compare
V(start) with the MC return of the same episode, which Exercise 1 printed.

In [ ]:
def td0_prediction(episodes, alpha, gamma, n_states, v0=0.0):
    """TD(0): for each transition (s, r, s2) in order, V[s] += alpha (target - V[s]).

    The target bootstraps from V[s2] unless s2 is terminal (the last state of an episode
    with terminated=True). A truncated episode's last transition still bootstraps.
    v0 is the starting value of every state.
    """
    V = np.full(n_states, float(v0))
    # TODO 2a: loop over episodes and their transitions; build the target with the terminal
    # rule, then move V[s] toward it.
    raise NotImplementedError(
        "TODO 2a: write td0_prediction, or run lab.use_reference('2a') to continue with the solution"
    )

<details><summary>Hint 2a</summary>

`zip(ep.states[:-1], ep.rewards, ep.states[1:], strict=True)` walks the transitions. The next state is
terminal only on the last transition of an episode whose `terminated` flag is True.

</details>

In [ ]:
#@title Solution 2a · td0_prediction (open after trying) { display-mode: "form" }
@lab.solution("2a")
def td0_prediction(episodes, alpha, gamma, n_states, v0=0.0):
    """TD(0): for each transition (s, r, s2) in order, V[s] += alpha (target - V[s])."""
    V = np.full(n_states, float(v0))
    for ep in episodes:
        last = len(ep.rewards) - 1
        for t, (s, r, s2) in enumerate(zip(ep.states[:-1], ep.rewards, ep.states[1:], strict=True)):
            terminal = ep.terminated and t == last
            target = r + gamma * (0.0 if terminal else V[s2])
            V[s] += alpha * (target - V[s])
    return V

In [ ]:
V_one = td0_prediction(episodes[:1], budget.alpha, GAMMA, S)
print(
    f"After one episode: V(cell above the goal) = {V_one[ABOVE_GOAL]:.4f}, V(start) = {V_one[START]:.4f}"
)
first_states = episodes[0].states[:-1]
print(
    f"  (your first episode visited the cell above the goal {first_states.count(ABOVE_GOAL)} "
    f"time(s) and the start {first_states.count(START)} time(s); each later visit moves V further)"
)
V_td = td0_prediction(episodes, budget.alpha, GAMMA, S)
td_rms = rms_on_route(V_td)
lab.metric("td_rms", round(td_rms, 4))
print(
    f"TD(0), alpha {budget.alpha}: V(start) = {V_td[START]:.3f} (exact {V_EXACT[START]:.3f}); "
    f"RMS on the route {td_rms:.3f} (MC: {mc_rms:.3f})"
)
fig, ax = plt.subplots()
ax.plot(V_EXACT[ROUTE], marker="o", label="exact (Lab 1)")
ax.plot(V_mc[ROUTE], marker="s", label="first-visit MC")
ax.plot(V_td[ROUTE], marker="^", label=f"TD(0), alpha {budget.alpha}")
ax.set(xlabel="position along the route (0 = start)", ylabel="V", title="MC and TD(0) on the route")
ax.legend()
plt.show()

**Explain.** Every target in your first episode was built from values still at 0, so the start moved only
α of the way toward −1, while MC's estimate from the same episode is the whole return. That
gap is the bias of the TD target (TD bias): which values must improve before the start's
target is right? After all the episodes, is your TD(0) error above or below MC's?

In [ ]:
lab.check("2a", m03.check_td0_prediction, td0_prediction, label="2a.1")

In [ ]:
lab.check("2a", m03.check_td0_rms, td0_prediction, label="2a.2")

## Exercise 2b · Bias² and variance over 10 seeds (5 min)

**Bias² and variance over 10 seeds.** Run a method on 10 seeds' worth of episodes and you get
10 estimates of each state's value. Their mean squared error against the exact V splits
exactly into the squared bias of their mean and their spread around that mean
(decomposition). Write the split; the provided cell then runs MC (1/N), constant-α MC and
TD(0) on the same 10 seeds and reports both terms for each.

**Predict.** Which method will have the lowest variance: MC (1/N), constant-α MC or TD(0)? Which will
have the largest bias²?

In [ ]:
def bias2_variance(estimates, v_true):
    """Squared bias and variance of estimates over seeds (decomposition).

    estimates has shape (seeds, states), one row per seed; v_true has shape (states,).
    Returns (bias2, variance): the mean over states of (mean over seeds - v_true)^2, and the
    mean over states of the spread over seeds around that mean (divide by the number of
    seeds, not seeds - 1).
    """
    estimates, v_true = np.asarray(estimates, dtype=float), np.asarray(v_true, dtype=float)
    # TODO 2b: two lines: the mean over seeds for each state, then the two averages over states.
    raise NotImplementedError(
        "TODO 2b: write bias2_variance, or run lab.use_reference('2b') to continue with the solution"
    )

<details><summary>Hint 2b</summary>

`estimates.mean(axis=0)` is one number per state. bias² averages `(mean - v_true) ** 2` over
states; the variance averages `estimates.var(axis=0)` over states (NumPy divides by the
number of seeds by default). Check yourself: the two add up to the average over seeds and
states of `(estimates - v_true) ** 2`.

</details>

In [ ]:
#@title Solution 2b · bias2_variance (open after trying) { display-mode: "form" }
@lab.solution("2b")
def bias2_variance(estimates, v_true):
    """Squared bias and variance of estimates over seeds (decomposition)."""
    estimates, v_true = np.asarray(estimates, dtype=float), np.asarray(v_true, dtype=float)
    mean = estimates.mean(axis=0)
    return float(np.mean((mean - v_true) ** 2)), float(np.mean(estimates.var(axis=0)))

In [ ]:
def mc_constant_alpha(episodes, alpha, gamma, n_states):
    """Provided: first-visit MC with a constant step size (the fair partner of TD(0) at alpha)."""
    V = np.zeros(n_states)
    for ep in episodes:
        G, returns = 0.0, [0.0] * len(ep.rewards)
        for t in reversed(range(len(ep.rewards))):
            G = ep.rewards[t] + gamma * G
            returns[t] = G
        seen = set()
        for t, s in enumerate(ep.states[:-1]):
            if s in seen:
                continue
            seen.add(s)
            V[s] += alpha * (returns[t] - V[s])
    return V


# 10 seeds per seat (seat 0 uses 0-9, seat 1 uses 100-109, ...), so the room's seeds differ.
seeds = [SEED * 100 + k for k in range(10)]
lab.seeds(seeds)
estimates = {"mc": [], "mcc": [], "td": []}
for k in seeds:
    eps_k = collect(env, PI, budget.episodes, seed=k)
    estimates["mc"].append(mc_prediction(eps_k, GAMMA, S)[ROUTE])
    estimates["mcc"].append(mc_constant_alpha(eps_k, budget.alpha, GAMMA, S)[ROUTE])
    estimates["td"].append(td0_prediction(eps_k, budget.alpha, GAMMA, S)[ROUTE])
names = {"mc": "MC (1/N)", "mcc": "MC (constant alpha)", "td": "TD(0)"}
print(f"{budget.episodes} episodes, alpha {budget.alpha}, 10 seeds; averages over the route states")
print(f"{'method':22s} {'bias^2':>8s} {'variance':>9s} {'sum':>8s} {'MSE':>8s}")
for key, rows in estimates.items():
    est = np.array(rows)
    bias2, var = bias2_variance(est, V_EXACT[ROUTE])
    mse = float(np.mean((est - V_EXACT[ROUTE]) ** 2))  # computed directly, to compare
    lab.metric(f"{key}_bias2", round(bias2, 4))
    lab.metric(f"{key}_var", round(var, 4))
    print(f"{names[key]:22s} {bias2:8.3f} {var:9.3f} {bias2 + var:8.3f} {mse:8.3f}")

**Explain.** TD(0)'s bias² is the largest in the table. Which quantity in (TD bias) is still large after
this many episodes, and why does it shrink as the budget grows? At the same α, is TD(0)'s
variance below constant-α MC's on your seat? Ten seeds give one rough number per method, and
some seats will see the order flip: write down your seat's order; the debrief counts how
many seats got each order before anyone concludes. (Gotcha: "TD is biased, so MC is
better.")

In [ ]:
lab.check("2b", m03.check_bias2_variance, bias2_variance)

## Exercise 3 · The n-step target (6 min)

**The n-step target.** Between TD(0) (n = 1) and Monte Carlo (n ≥ T) sits the n-step
return: n sampled rewards, then a bootstrap discounted by γⁿ (n-step). Two end cases: if
t + n reaches the end of a *terminated* episode, there is nothing to bootstrap from; if it
reaches the end of a *truncated* one, bootstrap from the last state with γ^(T − t).

**Predict.** An episode of 4 moves, each −1, that terminates. V = −2 for every nonterminal state and
γ = 0.9. With n = 2, what is the target for t = 0? And for t = 2, where the two steps run
into the terminal state?

In [ ]:
def n_step_target(rewards, states, t, n, V, gamma, terminated):
    """G_t^(n): rewards r_t..r_{t+n-1} discounted, plus gamma^n V[s_{t+n}] if t + n < T.

    rewards has T entries, states has T + 1 (the last is s_T). If t + n >= T, the sum runs to
    the end; then add gamma^(T - t) V[s_T] if the episode was truncated (not terminated),
    and nothing if it terminated.
    """
    # TODO 3: sum the rewards from t to min(t + n, T) - 1 with increasing powers of gamma,
    # then handle the three cases for the bootstrap term.
    raise NotImplementedError(
        "TODO 3: write n_step_target, or run lab.use_reference(3) to continue with the solution"
    )

<details><summary>Hint 3</summary>

Let `T = len(rewards)` and `end = min(t + n, T)`. The rewards run from `t` to `end - 1`. The
bootstrap, if there is one, is `V[states[end]]`, discounted by γ^(end − t). Now list the
three cases: `end < T`; `end == T` and terminated; `end == T` and truncated. Which of them
add the bootstrap?

</details>

In [ ]:
#@title Solution 3 · n_step_target (open after trying) { display-mode: "form" }
@lab.solution(3)
def n_step_target(rewards, states, t, n, V, gamma, terminated):
    """G_t^(n): rewards r_t..r_{t+n-1} discounted, plus gamma^n V[s_{t+n}] if t + n < T."""
    T = len(rewards)
    end = min(t + n, T)
    G = sum(gamma ** (k - t) * rewards[k] for k in range(t, end))
    if end < T or not terminated:
        G += gamma ** (end - t) * V[states[end]]
    return G

In [ ]:
corridor = Episode(
    states=[0, 1, 2, 3, 4],
    actions=[1, 1, 1, 1],
    rewards=[-1.0] * 4,
    terminated=True,
    truncated=False,
)
V_flat = np.full(5, -2.0)
print("Terminated 4-step episode, V = -2, gamma 0.9, n = 2:")
for t in range(4):
    print(
        f"  t = {t}: G^(2) = {n_step_target(corridor.rewards, corridor.states, t, 2, V_flat, 0.9, True):.4f}"
    )
cut = corridor._replace(terminated=False, truncated=True)
print(
    f"The same episode truncated instead: t = 2 gives {n_step_target(cut.rewards, cut.states, 2, 2, V_flat, 0.9, False):.4f}"
)
ep = episodes[0]
print(f"Your first episode (T = {len(ep.rewards)}), targets for t = 0 with your TD(0) values:")
for n in (1, 2, 4, 8, 16, 1000):
    print(
        f"  n = {n:4d}: {n_step_target(ep.rewards, ep.states, 0, n, V_td, GAMMA, ep.terminated):.4f}"
    )
print(f"  MC return of the episode:   {mc_prediction([ep], GAMMA, S)[START]:.4f}")

**Explain.** At n = 1 the target is TD(0)'s; once n reaches T it equals the Monte Carlo return, and
the truncated version keeps a bootstrap where the terminated one has none (n-step). As n
grows, which part of the target carries more sampled randomness, and which part carries
less of V's error? The stretch sweeps n and α to see where the two meet.

In [ ]:
lab.check(3, m03.check_n_step_target, n_step_target)

## Exercise 4 · SARSA (8 min)

**SARSA.** Now the policy is not fixed: act ε-greedily on Q while learning Q. SARSA's
target bootstraps from the action the agent actually takes next, Q(s', a') (SARSA), so it
learns the value of the ε-greedy policy itself, exploration included. `epsilon_greedy`,
a greedy-path extractor and a grid printer are provided.

In [ ]:
def epsilon_greedy(Q, s, epsilon, rng):
    """Provided: the greedy action with probability 1 - epsilon, else uniform (ties at random)."""
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(best[0]) if len(best) == 1 else int(rng.choice(best))


def greedy_path(Q, max_steps=None):
    """Provided: the states visited from the start by always taking argmax Q, until the goal,
    a revisit (a loop) or max_steps. Returns (path, reached_goal)."""
    s, path = START, [START]
    for _ in range(max_steps or S):
        if s == GOAL:
            return path, True
        s = int(np.argmax(mdp.P[s, int(np.argmax(Q[s]))]))
        if s in path:
            return path + [s], False
        path.append(s)
    return path, s == GOAL


def show_path(path, label):
    """Provided: draw the grid as text; '.' free, 'C' cliff, '*' the path, S start, G goal."""
    grid = [["." for _ in range(COLS)] for _ in range(ROWS)]
    for s in mdp.cliff:
        grid[s // COLS][s % COLS] = "C"
    for s in path:
        grid[s // COLS][s % COLS] = "*"
    grid[START // COLS][START % COLS], grid[GOAL // COLS][GOAL % COLS] = "S", "G"
    print(f"{label}: {len(path) - 1} moves")
    print("\n".join(" ".join(row) for row in grid))


def smooth(x, k=20):
    """Provided: moving average over k episodes, for plotting noisy online returns."""
    x = np.asarray(x, dtype=float)
    return np.convolve(x, np.ones(k) / k, mode="valid")

**Predict.** SARSA's greedy path will take some number of moves m (the run prints it). Over the last 500
episodes, will SARSA's online return per episode be above −m, about −m, or below −m? Why?

In [ ]:
def sarsa(env, episodes, alpha, epsilon, gamma, seed, q0=0.0):
    """SARSA on a Gymnasium env. Returns (Q, online), online = the reward sum of each episode.

    Start Q at q0 (a number, or an array to continue from). All randomness comes from seed:
    env.reset(seed=seed) once, and rng = np.random.default_rng(seed) for epsilon_greedy.
    """
    rng = np.random.default_rng(seed)
    Q = np.full((S, A), float(q0)) if np.isscalar(q0) else np.array(q0, dtype=float)
    online = []
    env.reset(seed=seed)
    # TODO 4: for each episode: reset, pick a; loop: step, pick a2 from s2, update Q[s, a]
    # toward r + gamma Q[s2, a2] (0 if terminated), move on; stop on terminated or truncated.
    raise NotImplementedError(
        "TODO 4: write sarsa, or run lab.use_reference(4) to continue with the solution"
    )

<details><summary>Hint 4</summary>

The next action a2 is chosen before the update and then *used* as the next step's action:
that is what makes it on-policy. The bootstrap is `0.0 if terminated else Q[s2, a2]`;
`truncated` only ends the episode.

</details>

In [ ]:
#@title Solution 4 · sarsa (open after trying) { display-mode: "form" }
@lab.solution(4)
def sarsa(env, episodes, alpha, epsilon, gamma, seed, q0=0.0):
    """SARSA on a Gymnasium env. Returns (Q, online), online = the reward sum of each episode."""
    rng = np.random.default_rng(seed)
    Q = np.full((S, A), float(q0)) if np.isscalar(q0) else np.array(q0, dtype=float)
    online = []
    env.reset(seed=seed)
    for _ in range(episodes):
        s, _ = env.reset()
        a = epsilon_greedy(Q, s, epsilon, rng)
        total, done = 0.0, False
        while not done:
            s2, r, terminated, truncated, _ = env.step(a)
            a2 = epsilon_greedy(Q, s2, epsilon, rng)
            target = r + gamma * (0.0 if terminated else Q[s2, a2])
            Q[s, a] += alpha * (target - Q[s, a])
            total += r
            s, a, done = s2, a2, terminated or truncated
        online.append(total)
    return Q, np.array(online)

In [ ]:
Q_sarsa, online_sarsa = sarsa(
    env, budget.control_episodes, budget.control_alpha, EPSILON, GAMMA, seed=SEED
)
path_sarsa, reached = greedy_path(Q_sarsa)
show_path(path_sarsa, "SARSA's greedy path" + ("" if reached else " (does not reach the goal)"))
edge = [s for s in path_sarsa if s // COLS == ROWS - 2 and 1 <= s % COLS <= COLS - 2]
print(f"Cells of the path directly above the cliff: {len(edge)}")
print(
    f"Online return, mean of the last {budget.window} episodes: {online_sarsa[-budget.window :].mean():.1f}"
)
fig, ax = plt.subplots()
ax.plot(smooth(online_sarsa), label="SARSA (moving average of 20)")
ax.set(xlabel="episode", ylabel="reward sum per episode", title="Online return while learning")
ax.legend()
plt.show()

**Explain.** A non-greedy "down" fires with probability ε/4 = 0.025 and costs 100, so an edge cell is
worth about 2.5 less per visit than one farther up, under the policy SARSA evaluates. Which
of the two bootstraps, Q(s', a') (SARSA) or max Q(s', ·), is the one that sees this cost,
and why? Read your path: how many rows up does it go before heading right?

In [ ]:
lab.check(4, m03.check_sarsa, sarsa)

## Exercise 5 · Q-learning and the cliff contrast (8 min)

**Q-learning and the cliff contrast.** One symbol changes: the target bootstraps from
max over a' of Q(s', a'), whatever action the agent takes next (Q-learning). It estimates
Q* no matter how it explores. Then compare the two methods on your seed: the greedy paths,
and the online returns earned by their ε-greedy behavior while learning.

**Predict.** By about how much per episode will Q-learning's online return over the last 500 episodes be
below SARSA's: about 3, about 30, or about 300? Use the briefing's chance of a fall on one
traverse of the edge, 0.24.

In [ ]:
def q_learning(env, episodes, alpha, epsilon, gamma, seed, q0=0.0):
    """Q-learning on a Gymnasium env. Returns (Q, online), online = the reward sum per episode.

    Same conventions as sarsa: start at q0; env.reset(seed=seed) once; rng from seed.
    """
    rng = np.random.default_rng(seed)
    Q = np.full((S, A), float(q0)) if np.isscalar(q0) else np.array(q0, dtype=float)
    online = []
    env.reset(seed=seed)
    # TODO 5: as in sarsa, but the target uses Q[s2].max() (0 if terminated), and the next
    # action is chosen from s2 after the update.
    raise NotImplementedError(
        "TODO 5: write q_learning, or run lab.use_reference(5) to continue with the solution"
    )

<details><summary>Hint 5</summary>

Choose `a = epsilon_greedy(Q, s, ...)` at the top of each step, take it, then update with
`Q[s2].max()`. The action chosen next is not part of the target.

</details>

In [ ]:
#@title Solution 5 · q_learning (open after trying) { display-mode: "form" }
@lab.solution(5)
def q_learning(env, episodes, alpha, epsilon, gamma, seed, q0=0.0):
    """Q-learning on a Gymnasium env. Returns (Q, online), online = the reward sum per episode."""
    rng = np.random.default_rng(seed)
    Q = np.full((S, A), float(q0)) if np.isscalar(q0) else np.array(q0, dtype=float)
    online = []
    env.reset(seed=seed)
    for _ in range(episodes):
        s, _ = env.reset()
        total, done = 0.0, False
        while not done:
            a = epsilon_greedy(Q, s, epsilon, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            target = r + gamma * (0.0 if terminated else Q[s2].max())
            Q[s, a] += alpha * (target - Q[s, a])
            total += r
            s, done = s2, terminated or truncated
        online.append(total)
    return Q, np.array(online)

In [ ]:
Q_q, online_q = q_learning(
    env, budget.control_episodes, budget.control_alpha, EPSILON, GAMMA, seed=SEED
)
path_q, reached_q = greedy_path(Q_q)
show_path(path_q, "Q-learning's greedy path" + ("" if reached_q else " (does not reach the goal)"))
show_path(path_sarsa, "SARSA's greedy path")
mean_sarsa = float(online_sarsa[-budget.window :].mean())
mean_q = float(online_q[-budget.window :].mean())
lab.metric("sarsa_online", round(mean_sarsa, 2))
lab.metric("q_online", round(mean_q, 2))
lab.metric("sarsa_lead", round(mean_sarsa - mean_q, 2))
lab.metric("sarsa_path_moves", len(path_sarsa) - 1 if reached else None)  # None: a loop
lab.metric("q_path_moves", len(path_q) - 1 if reached_q else None)
print(f"Online return over the last {budget.window} episodes, your seed {SEED}:")
print(
    f"  SARSA {mean_sarsa:7.1f}   Q-learning {mean_q:7.1f}   difference {mean_sarsa - mean_q:+.1f}"
)
print("Single seed: a fact to put beside the threshold and the room's pooled seeds, not a ranking.")
fig, ax = plt.subplots()
ax.plot(smooth(online_sarsa), label="SARSA")
ax.plot(smooth(online_q), label="Q-learning")
ax.set(
    xlabel="episode",
    ylabel="reward sum per episode (moving average of 20)",
    title=f"The cliff contrast, epsilon {EPSILON}, alpha {budget.control_alpha}, seed {SEED}",
)
ax.legend()
plt.show()

**Explain.** Q-learning's greedy path is the 13-move edge route, SARSA's is longer, and yet SARSA collects
more reward while learning. Both agents explore with the same ε: whose exploration is
expensive, and why does Q-learning's target not care? If ε were decayed to 0, what would
happen to the gap? (Gotcha: "Q-learning is better than SARSA.")

In [ ]:
lab.check(5, m03.check_q_learning_path, q_learning, label="5.1")

In [ ]:
lab.check(5, m03.check_cliff_contrast, sarsa, q_learning, label="5.2")

## Exercise 6 · Planted bug: bootstrapping through the terminal (8 min)

**Planted bug.** A colleague wrote `q_learning_suspect`. It runs, its greedy path is the
optimal route, and it passes any test that starts Q at 0. It is wrong. Train it from an
optimistic start, Q₀ = 10, and read three things against what they must be: the return it
earns while learning, max Q at the start against V*(start) = −(1 − γ¹³)/(1 − γ) ≈ −9.73 (the
13-move route), and Q of its last move (the cell above the goal, action down). Find the
line, fix it.

**Predict.** With Q₀ = 10 and γ = 0.95, what should Q(cell above the goal, down) converge to in a correct
learner? Can any return in this world, from any state, be above −1? Write both, then run the
cell below.

In [ ]:
def q_learning_suspect(env, episodes, alpha, epsilon, gamma, seed, q0=0.0):
    """A colleague's Q-learning. It learns the optimal path. Is it right?"""
    # TODO 6: once the diagnostics below show the problem, find the one wrong line and fix it.
    # Then rerun this cell and the run cell.
    rng = np.random.default_rng(seed)
    Q = np.full((S, A), float(q0)) if np.isscalar(q0) else np.array(q0, dtype=float)
    online = []
    env.reset(seed=seed)
    for _ in range(episodes):
        s, _ = env.reset()
        total, done = 0.0, False
        while not done:
            a = epsilon_greedy(Q, s, epsilon, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            target = r + gamma * Q[s2].max()
            Q[s, a] += alpha * (target - Q[s, a])
            total += r
            s, done = s2, terminated or truncated
        online.append(total)
    return Q, np.array(online)

<details><summary>Hint 6</summary>

The suspect's last move settled at −1 + γ · 10. Which Q-values in the grid are still 10
after training, and where in the target line could one of them come in?

</details>

In [ ]:
#@title Solution 6 · q_learning_suspect (open after trying) { display-mode: "form" }
@lab.solution(6)
def q_learning_suspect(env, episodes, alpha, epsilon, gamma, seed, q0=0.0):
    """The colleague's Q-learning, fixed: no bootstrap through a terminal state."""
    rng = np.random.default_rng(seed)
    Q = np.full((S, A), float(q0)) if np.isscalar(q0) else np.array(q0, dtype=float)
    online = []
    env.reset(seed=seed)
    for _ in range(episodes):
        s, _ = env.reset()
        total, done = 0.0, False
        while not done:
            a = epsilon_greedy(Q, s, epsilon, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            target = r + gamma * (0.0 if terminated else Q[s2].max())
            Q[s, a] += alpha * (target - Q[s, a])
            total += r
            s, done = s2, terminated or truncated
        online.append(total)
    return Q, np.array(online)

In [ ]:
V_STAR_START = -(1 - GAMMA**13) / (1 - GAMMA)  # the optimal route has 13 moves
Q0 = 10.0


def value_curve(learner, episodes, chunk, seed, q0):
    """Provided: train in chunks; record max_a Q(start, a) after each chunk and every episode's
    online return. Returns (xs, values, online, Q)."""
    Q, xs, values, online = q0, [], [], []
    for i in range(episodes // chunk):
        Q, on = learner(
            env, chunk, budget.control_alpha, EPSILON, GAMMA, seed=seed * 1000 + i, q0=Q
        )
        online.extend(on)
        xs.append((i + 1) * chunk)
        values.append(Q[START].max())
    return xs, values, np.array(online), Q


chunk = max(10, budget.control_episodes // 50)
xs, v_suspect, on_suspect, Q_suspect = value_curve(
    q_learning_suspect, budget.control_episodes, chunk, SEED, Q0
)
_, v_fixed, on_fixed, Q_fixed = value_curve(q_learning, budget.control_episodes, chunk, SEED, Q0)
print(f"Optimistic start Q0 = {Q0}; after {xs[-1]} episodes:")
print(
    f"  suspect: Q(above goal, down) = {Q_suspect[ABOVE_GOAL, DOWN]:8.4f}   max Q(start) = {Q_suspect[START].max():8.4f}"
)
print(
    f"  yours:   Q(above goal, down) = {Q_fixed[ABOVE_GOAL, DOWN]:8.4f}   max Q(start) = {Q_fixed[START].max():8.4f}"
)
print(f"  truth:   Q*(above goal, down) = {-1.0:8.4f}   V*(start) = {V_STAR_START:8.4f}")
print(
    f"  Q(goal, ·) in your q_learning: {Q_fixed[GOAL]} (never updated: no action is taken in "
    "the goal, and with the mask no target reads it)"
)
show_path(greedy_path(Q_suspect)[0], "suspect's greedy path")
fig, (ax_r, ax_v) = plt.subplots(1, 2, figsize=(10, 3.6))
ax_r.plot(smooth(on_suspect), label="q_learning_suspect")
ax_r.plot(smooth(on_fixed), label="your q_learning")
ax_r.set(xlabel="episode", ylabel="online return (moving average of 20)", title="What it earns")
ax_r.legend()
ax_v.plot(xs, v_suspect, label="q_learning_suspect")
ax_v.plot(xs, v_fixed, label="your q_learning")
ax_v.axhline(V_STAR_START, linestyle="--", color="gray", label="V*(start)")
ax_v.set(xlabel="episodes", ylabel="max Q(start, a)", title=f"What it believes, from Q0 = {Q0}")
ax_v.legend()
plt.show()

**Explain.** Before your fix, the suspect's last move settled at −1 + γ · 10 = 8.5 and its start at about
−4.6, both above the truth, yet its greedy path was optimal: a bonus that shrinks with
distance cannot reorder the routes. Look at the left panel first: could you have found this
bug from the returns? (Gotcha: "My returns went up, so it works.") Why did the same bug
change nothing when Q started at 0, and would Sutton and Barto's choice, Q(terminal, ·) = 0
instead of a mask, have fixed it here? (terminal mask) Would `0.0 if (terminated or
truncated)` also pass this checkpoint, and why is it still wrong in CartPole? (Gotcha:
"`done` means stop bootstrapping.")

**Diagnostics for this lab.** For a learner, plot max Q(start) over episodes against a known
V*(start), and print Q of the last move against its reward: a value curve that flattens at
the wrong level, or a last move worth more than its reward, means the target is wrong, not
the exploration. The return curve cannot show this: it measures the policy you follow, not
the values you learned. Run every such test from an optimistic start as well as from zero;
the terminal bug is invisible from zero. For a predictor, plot the estimate along the route
against the exact V and read the error's shape: a constant offset points to the discount or
the terminal rule, an offset that grows with distance from the goal points to a bootstrap
that forgets its start too slowly (small α, few episodes), and a ragged curve with a correct
mean points to variance, which more episodes or more seeds reduce. A greedy path that loops
means Q is noisy where the agent rarely goes; it is not a bug.

In [ ]:
lab.check(6, m03.check_q_learning_suspect, q_learning_suspect)

**Your room code.** The last cell, `lab.finish()`, prints one line that starts with `ROOM m03`.
It holds your seat, the checkpoints you passed, whether QUICK budgets ran, and the numbers
the debrief pools: `mc_rms` and `td_rms` (Exercises 1 and 2a), the bias² and variance of each
method (2b), and SARSA's and Q-learning's online returns, SARSA's lead and the greedy-path
moves (5). Paste it where your instructor says.

In [ ]:
# GENERATED by scripts/build_notebooks.py. Prints your summary and your run record.
lab.finish()

<!-- GENERATED by scripts/build_notebooks.py from _variables.yml. Do not edit this cell. -->
**Next:** [Module 4 · Bandits from first principles](https://project-delphi.github.io/practical-rl/modules/04-bandits.html). First, paste your room code where your instructor says.

## Stretch (optional) · beyond the core path

Nothing later depends on this section.

**The n-step dial on your seed.** Build n-step TD prediction on your `n_step_target`: for
each episode and each t in order, move V[s_t] a fraction α toward G_t^(n), computed with the
values as they stand (the briefing's n-step update; Sutton and Barto's Eq. 7.2 makes the same
updates in the same order, n steps later in time). Then the provided sweep runs n in
{1, 2, 4, 8, 16} and several α on 5 seeds and plots the RMS error on the route: the
briefing's figure, on your seat's episodes.

In [ ]:
def n_step_td_prediction(episodes, alpha, gamma, n_states, n):
    """n-step TD: V[s_t] += alpha (G_t^(n) - V[s_t]) for every t of every episode, in order."""
    # Stretch: loop over episodes and t; call n_step_target with the current V.
    raise NotImplementedError(
        "Stretch: write n_step_td_prediction, or run lab.use_reference('stretch') to see the solution"
    )

In [ ]:
#@title Stretch solution · n_step_td_prediction (open after trying) { display-mode: "form" }
@lab.solution("stretch")
def n_step_td_prediction(episodes, alpha, gamma, n_states, n):
    """n-step TD: V[s_t] += alpha (G_t^(n) - V[s_t]) for every t of every episode, in order."""
    V = np.zeros(n_states)
    for ep in episodes:
        for t, s in enumerate(ep.states[:-1]):
            G = n_step_target(ep.rewards, ep.states, t, n, V, gamma, ep.terminated)
            V[s] += alpha * (G - V[s])
    return V

In [ ]:
lab.check("stretch", m03.check_n_step_td_prediction, n_step_td_prediction, label="stretch")
ns, alphas = [1, 2, 4, 8, 16], [0.02, 0.05, 0.1, 0.2]
stretch_seeds = [SEED * 100 + 50 + k for k in range(5)]
episodes_by_seed = [collect(env, PI, budget.episodes, seed=k) for k in stretch_seeds]
fig, ax = plt.subplots()
for alpha in alphas:
    errs = np.array(
        [
            [rms_on_route(n_step_td_prediction(eps_k, alpha, GAMMA, S, n)) for n in ns]
            for eps_k in episodes_by_seed
        ]
    )
    ax.plot(ns, errs.mean(axis=0), marker="o", label=f"alpha {alpha} (mean of 5 seeds)")
    ax.fill_between(ns, errs.min(axis=0), errs.max(axis=0), alpha=0.15)
ax.set(
    xscale="log",
    xlabel="n",
    ylabel="RMS error on the route",
    title=f"n-step TD after {budget.episodes} episodes",
)
ax.set_xticks(ns, [str(n) for n in ns])
ax.legend()
plt.show()

<!-- GENERATED by scripts/build_notebooks.py from _variables.yml. Do not edit this cell. -->
**Next:** [Module 4 · Bandits from first principles](https://project-delphi.github.io/practical-rl/modules/04-bandits.html). First, paste your room code where your instructor says.